# Lesson 4 : Tools

Microsoft Agent Framework provides a wide variety of built-in tool's object (such as, Code Interpreter, Web Search, File Search, MCP tools, Browser Automation, etc), and you can use these useful tools in your agent.  
With ```FoundryChatClient``` in Microsoft Agent Framework, you can use the following 3 types of tools. :

- **Hosted tools** : As I have mentioned in Lesson 1, ```FoundryChatClient``` uses Azure AI Projects SDK (```azure-ai-projects```) v2. You can work with tools natively hosted in Azure AI Projects SDK. The available tools in this type will vary depending on the type of client. For example, you can use [Claude's web search tool](https://platform.claude.com/docs/en/agents-and-tools/tool-use/web-search-tool) when ```AnthropicClient```.
- **Foundry tools** : Microsoft Foundry provides various additional tools in the gallery (catalog) - such as, SharePoint tool, Fabric data agent tool, OpenAPI-integrated tool, or 3rd-party tools, and you can also use these pre-configured additional tools in ```FoundryChatClient```.
- **Tools in Microsoft Agent Framework (MAF)** : The library of Agent Framework SDK also provides native tools. Unlike above server-side tool calling, these native tools are mostly handled as local functions in LLM invocation, and these are processed in Microsoft Agent Framework SDK which runs on your local computers. For example, native MCP tools in Agent Framework (such as, ```MCPStdioTool```, ```MCPStreamableHTTPTool```, or ```MCPWebsocketTool```) are locally processed in Microsoft Agent Framework along with MCP protocol specification.

In this exercise, we will explore these 3 types of tools with examples of web search tool and MCP tool.

## Hosted tool (Web Search example)

In the first example, we explore web search tool in ```FoundryChatClient``` hosted tools.

Firstly, same as in Lesson 1, we create a client as follows.

In [1]:
from dotenv import load_dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity.aio import AzureCliCredential

load_dotenv()

credential = AzureCliCredential()
client = FoundryChatClient(credential=credential)

Now we create tool definition for native web search tool in Microsoft Foundry, and create an agent with this tool setting.

By calling ```get_web_search_tool()``` method in ```FoundryChatClient```, ```WebSearchPreviewTool``` object (in Azure AI Projects SDK) is internally created and used.

In [2]:
from agent_framework import Agent

web_search_tool = client.get_web_search_tool()
agent = Agent(
    name="WeatherAgentWithSearchTool",
    client=client,
    instructions="You are an agent about weather information.",
    tools=[web_search_tool])

Now we run the agent as follows.

As you see, this agent knows "what day is it today" or "what is the actual weather condition today", because web search is performed internally.

In Lesson 1, the tool execution is run on your local function. In this example, however, Microsoft Foundry will handle this process on server.

In [3]:
from IPython.display import Markdown, display

result = await agent.run("Tell me the weather and temperature in Osaka today.")
display(Markdown(result.text))

Today in **Osaka (Fri, Jan 16, 2026)**: **mostly sunny**.

- **Temperature:** around **15°C / 59°F** high, **~5°C / 41°F** low [Osaka-shi, Osaka, Japan Weather Forecast | AccuWeather](https://www.accuweather.com/en/jp/osaka-shi/225007/weather-forecast/225007)[10 Day Weather - Osaka, Osaka, Japan - The Weather Channel](https://weather.com/weather/tenday/l/Osaka+Osaka+Japan?placeId=441174f51a1951566e6b1d02bd724effab80d7359e5f241540d1aa46dfecc59f)[Osaka, Japan 14 day weather forecast - timeanddate.com](https://www.timeanddate.com/weather/japan/osaka/ext)[Weather - Osaka City - 14-Day Forecast & Rain | Ventusky](https://www.ventusky.com/osaka)[Osaka Weather Forecast](https://www.weather-forecast.com/locations/Osaka/forecasts/latest)

## 1. Hosted tool (MCP example)

Next we explore MCP tool hosted in Microsoft Foundry client on Microsoft Agent Framework. (This will internally use MCP tool definition in Azure OpenAI Responses API.)

Same as above example, we call built-in ```get_mcp_tool()``` method in ```FoundryChatClient``` to get ```MCPTool``` object.

In this example, we create an agent to answer Microsoft technical questions.<br>
This agent uses a remote MCP server (Streamable HTTP server), which provides information about Microsoft Learn document.

In [4]:
mcp_tool = client.get_mcp_tool(
    name="Microsoft Learn MCP",
    url="https://learn.microsoft.com/api/mcp",
    approval_mode="never_require",
)
agent = Agent(
    name="MSTechKnowledgeAgent",
    client=client,
    instructions="You are an agent who answers technical questions about Microsoft products and services.",
    tools=[mcp_tool],
)

Let's ask a technical question about Microsoft Azure.  
In this call, MCP tool calling (about Microsoft Learn document) is handled in Microsoft Foundry. (You can verify that MCP tool is being used internally by checking the internal steps by [tracing](./02_trace.ipynb).)

In [5]:
result = await agent.run("How to create an Azure storage account using Azure CLI ?")
display(Markdown(result.text))

To create an Azure Storage account with Azure CLI:

1) Sign in (if you’re running CLI locally)
```bash
az login
```

2) Create (or reuse) a resource group
```bash
az group create \
  --name storage-rg \
  --location eastus
```

3) Create the storage account (general-purpose v2)
```bash
az storage account create \
  --name <uniqueStorageAccountName> \
  --resource-group storage-rg \
  --location eastus \
  --sku Standard_RAGRS \
  --kind StorageV2 \
  --min-tls-version TLS1_2 \
  --allow-blob-public-access false
```

Notes:
- `<uniqueStorageAccountName>` must be **globally unique** in Azure (3–24 chars, lowercase letters and numbers only).
- Common SKUs: `Standard_LRS`, `Standard_GRS`, `Standard_RAGRS`, `Standard_ZRS`, etc.

Reference: `az storage account create` and the “Create an Azure storage account” guide on Microsoft Learn:  
https://learn.microsoft.com/azure/storage/common/storage-account-create#create-a-storage-account

## 2. Foundry tool (MCP example)

In the next example, we use additional Foundry tools in Microsoft Agent Framework.<br>
Foundry tools provides various added values - such as, built-in authentication, toolbox (tool's set for sharing and reuse), guardrails for tool calling, etc.

> Note : Here I don't go details about Foundry toolbox, but **Foundry toolbox** is also an MCP server that has an endpoint URL, so you can handle it in the same way as a regular MCP call using the following ```MCPStreamableHTTPTool```.

In this example, we change above MCP example to use built-in "Microsoft Learn MCP server" tool in Foundry.

### Preparation

Before writing code, please connect to "Microsoft Learn" tool in Microsoft Foundry UI as follows.

1. Open Foundry Portal.
2. Go to "Build" tab.
3. Select "Tools" menu.
4. Select "Microsoft Learn MCP server" in catalog, and establish connection.

After the connection is established, please **copy the project connection id**.

### Run code

Now let's create an agent with this Foundry tool as follows.<br>
In the following code, **please replace the following ```PROJECT_CONNECTION_ID```** with project connection id that you have obtained above.

> Note : All tools registered in your Foundry project has unique project connection id, and your agents built in Microsoft Agent Framework can then connect to any tools in project by setting this id. (Mostly 3rd party tools in Microsoft Foundry has MCP type.)

In [6]:
# ToDo : fill your below settings
#       (e.g., /subscriptions/{AZURE_SUBSCRIPTION_ID}/resourceGroups/{RESOURCE_GROUP_NAME}/providers/Microsoft.CognitiveServices/accounts/{FOUNDRY_RESOURCE_NAME}/projects/{FOUNDRY_PROJECT_NAME}/connections/{CONNECTED_RESOURCE_NAME})
PROJECT_CONNECTION_ID = "xxxxxxxxxx"

agent = Agent(
    name="MSTechKnowledgeAgent",
    client=client,
    instructions="You are an agent who answers technical questions about Microsoft products and services.",
    tools=[
        {
            "type": "mcp",
            "server_label": "mymcp01",
            "server_url": "https://learn.microsoft.com/api/mcp",
            "require_approval": "never",
            "project_connection_id": PROJECT_CONNECTION_ID,
        }
    ],
)

Same as above, let's ask a technical question about Microsoft Azure. (You can verify that MCP tool is being used internally by checking the internal steps by [tracing](./02_trace.ipynb).)

In [7]:
result = await agent.run("How to create an Azure storage account using Azure CLI ?")
display(Markdown(result.text))

Use **az storage account create** (storage accounts are Azure Resource Manager resources, so you’ll typically create/use a resource group first).

```azurecli
# Sign in (skip if already signed in, e.g., Cloud Shell)
az login

# 1) Create a resource group
az group create \
  --name storage-rg \
  --location eastus

# 2) Create a StorageV2 (general-purpose v2) storage account
# Note: storage account name must be globally unique (3-24 lowercase letters/numbers)
az storage account create \
  --name <account-name> \
  --resource-group storage-rg \
  --location eastus \
  --sku Standard_RAGRS \
  --kind StorageV2 \
  --min-tls-version TLS1_2 \
  --allow-blob-public-access false
```

(Optional) Verify:

```azurecli
az storage account show --resource-group storage-rg --name <account-name>
```

Refs:
- https://learn.microsoft.com/azure/storage/common/storage-account-create#create-a-storage-account
- https://learn.microsoft.com/cli/azure/storage/account?view=azure-cli-latest#az-storage-account-create

### [Optional] Work with MCP authentication in Foundry tools

In Foundry custom MCP tools, you can configure authentication.<br>
Especially, when you configure **OAuth identity passthrough** (see below picture), your client should handle the authentication in your code - such as, obtaining an authentication request, requesting authentication to user (display the login screen), and passing the obtained token to the MCP tool.

The procedure is as follows. :  
First, add your custom MCP tools in Microsoft Foundry with OAuth identity passthrough in authentication configuration.  
For instructions, please see [this official document](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/mcp-authentication).

![OAuth identity passthrough](./assets/foundry_oauth.png)

Run the following code to connect and handle authentication.  
Foundry agent returns a consent request as an event (with "```oauth_consent_request```" type) and a login URL, so please capture that event and displays the login screen to user. (Internally, your request is rejected and it returns a consent request.)

In [ ]:
PROJECT_CONNECTION_ID = "<FILL-PROJECT-CONNECTION-ID>"
MCP_SERVER_ENDPOINT = "<FILL-MCP-SERVER-ENDPOINT-URL>"
SYSTEM_PROMPT = "<FILL-INSTRUCTION-PROMPT>"
USER_PROMPT =   "<FILL-USER-PROMPT>"

# create agent
agent = Agent(
    name="CustomMCPWithAuthAgent",
    client=client,
    instructions=SYSTEM_PROMPT,
    tools=[
        {
            "type": "mcp",
            "server_label": "test01",
            "server_url": MCP_SERVER_ENDPOINT,
            "require_approval": "never",
            "project_connection_id": PROJECT_CONNECTION_ID,
        }
    ],
)

# get OAuth login requests
# (make sure to run with session
session = agent.create_session()

consent_links = []
async def run_to_get_consent():
    async for chunk in agent.run(
        USER_PROMPT,
        session=session,
        stream=True):
        for r in chunk.user_input_requests:
            if r.type == "oauth_consent_request":
                consent_links.append(r.consent_link)

await run_to_get_consent()

# ToDo :
# Show consent_link (URL for login) in your browser to be prompted to log in
for l in consent_links:
    print(l)

After the user has logged in, run the same request with the same session.  
By using the same session, ```previous_response_id``` is passed to Foundry agent and it recognizes that the user has logged in successfully.

In [ ]:
async def run_again():
    async for chunk in agent.run(
        USER_PROMPT,
        session=session,
        stream=True):
        if chunk.text:
            print(chunk.text, end="", flush=True)
    print("\n")

await run_again()

## 3. Tools in MAF (MCP example)

Microsoft Agent Framework also provides native built-in MCP tools - such as, ```MCPStdioTool```, ```MCPStreamableHTTPTool```, and ```MCPWebsocketTool```.<br>
Unlike above tools, these built-in tools are processed as local functions (i.e., **client-side tool calling**) in Microsoft Agent Framework SDK, in accordance with MCP protocol specifications.  

This method might be useful when MCP (or some part of MCP specification) is not supported in your client. For example, some remote API models might not support MCP STDIO-based server tools, but ```MCPStdioTool``` can do.<br>
In other cases, this method can be used for customizing the MCP invocation - e.g., updating MCP authentication headers per run.

In this example, we change above MCP example to use ```MCPStreamableHTTPTool``` in Microsoft Agent Framework (MAF) as follows. (You can verify that MCP tool is being used internally by checking the internal steps by [tracing](./02_trace.ipynb).)

In [8]:
from agent_framework import MCPStreamableHTTPTool

mcp_tool = MCPStreamableHTTPTool(
    name="Microsoft Learn MCP",
    url="https://learn.microsoft.com/api/mcp",
    load_prompts=False,
    approval_mode="never_require",
)

agent = Agent(
    name="MSTechKnowledgeAgent",
    client=client,
    instructions="You are an agent who answers technical questions about Microsoft products and services.",
    tools=[mcp_tool],
)

In [9]:
result = await agent.run("How to create an Azure storage account using Azure CLI ?")
display(Markdown(result.text))

1) Sign in (if you’re running locally):
```bash
az login
```

2) Create a resource group:
```bash
az group create \
  --name storage-resource-group \
  --location eastus
```

3) Create the storage account (general-purpose v2):
```bash
az storage account create \
  --name <account-name> \
  --resource-group storage-resource-group \
  --location eastus \
  --sku Standard_RAGRS \
  --kind StorageV2 \
  --min-tls-version TLS1_2 \
  --allow-blob-public-access false
```

Notes:
- `<account-name>` must be **globally unique**, **3–24 characters**, **lowercase letters and numbers only**.
- Choose a different redundancy SKU if needed (for example `Standard_LRS`, `Standard_GRS`, `Standard_ZRS`, etc.).

Reference: https://learn.microsoft.com/azure/storage/common/storage-account-create#create-a-storage-account